# Parte 1 · Pesos del bosque, verificación y proximidad (1.4, 1.5 y 1.6)

**Autor:** Herman Echeverría · Grupo Taiwan

**Declaración de uso de IA:** se utilizó Claude (Anthropic) como herramienta de apoyo para comprender lo que pedía cada inciso del enunciado y como asistencia al escribir y depurar el código. El análisis, la verificación de resultados y su interpretación fueron revisados por el autor.

Datos: California Housing, estandarizados como en 0.1. Los pesos del bosque se calculan con `src/bosque.py` y los de los otros cuatro modelos con `src/pesos.py`.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble._forest import _generate_sample_indices   # solo para explicar 1.4 con bootstrap

from datos import cargar_california
from pesos import pesos_ols, pesos_ridge, pesos_knn, pesos_arbol
from bosque import pesos_bosque, proximidad, verificar

X_train, X_test, y_train, y_test, _ = cargar_california()
q = X_test.iloc[:20]
n = len(X_train)

## 1.4 · Random forest

> **Pregunta 1.4 (a).** Con `RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.33, bootstrap=False, random_state=0)`, muestren que $w_i(x) = \frac{1}{T}\sum_{t=1}^{T} \frac{\mathbf 1\{x_i \text{ cae en la misma hoja que } x \text{ en el árbol } t\}}{\text{número de filas de entrenamiento en esa hoja}}$.

Sea $L_t(x)$ la hoja a la que el árbol $t$ manda a $x$. Cada árbol es un árbol de regresión, así que predice el promedio de las $y$ de entrenamiento que cayeron en esa hoja (igual que en 1.3). Con `bootstrap=False` cada árbol se ajusta con **todas** las filas de entrenamiento, así que

$$\hat y_t(x) = \sum_{i=1}^{n} \frac{\mathbf 1\{x_i \in L_t(x)\}}{|L_t(x)|}\, y_i .$$

El bosque promedia los $T$ árboles. Intercambiando las dos sumas:

$$\hat y(x) = \frac{1}{T}\sum_{t=1}^{T} \hat y_t(x) = \sum_{i=1}^{n} \underbrace{\frac{1}{T}\sum_{t=1}^{T} \frac{\mathbf 1\{x_i \in L_t(x)\}}{|L_t(x)|}}_{w_i(x)}\, y_i .$$

Es un promedio de $T$ vectores de pesos de árbol; cada uno es no negativo y suma 1, así que el promedio también. `max_features=0.33` solo cambia qué cortes se consideran, no la forma de los pesos.

In [ ]:
bosque = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.33,
                               bootstrap=False, random_state=0, n_jobs=-1).fit(X_train, y_train)
W_bosque = pesos_bosque(bosque, X_train, q)
pd.Series({"error máx. |Wy − predict|": verificar(W_bosque, y_train, bosque.predict(q)),
           "Σw mínima": W_bosque.sum(1).min(), "Σw máxima": W_bosque.sum(1).max()})

> **Pregunta 1.4 (b).** Prueben la misma fórmula con `bootstrap=True`. ¿Por qué ya no coincide?

In [ ]:
bosque_bs = RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.33,
                                  bootstrap=True, random_state=0, n_jobs=-1).fit(X_train, y_train)
W_bs = pesos_bosque(bosque_bs, X_train, q)
error_bs = np.abs(W_bs @ y_train - bosque_bs.predict(q))

# La fórmula corregida: en cada árbol, cada fila vota tantas veces como aparece en su muestra bootstrap
hojas_train, hojas_q = bosque_bs.apply(X_train), bosque_bs.apply(q)
W_corr = np.zeros((len(q), n))
for t, arbol in enumerate(bosque_bs.estimators_):
    veces = np.bincount(_generate_sample_indices(arbol.random_state, n, n), minlength=n)
    misma = (hojas_q[:, [t]] == hojas_train[None, :, t]) * veces
    W_corr += misma / misma.sum(1, keepdims=True)
W_corr /= len(bosque_bs.estimators_)

pd.Series({"error máx. con la fórmula de 1.4": error_bs.max(),
           "error medio con la fórmula de 1.4": error_bs.mean(),
           "error máx. contando las repeticiones del bootstrap": verificar(W_corr, y_train, bosque_bs.predict(q)),
           "filas que no vio un árbol (promedio)": np.mean([np.sum(np.bincount(_generate_sample_indices(a.random_state, n, n), minlength=n) == 0) / n
                                                            for a in bosque_bs.estimators_])})

Con `bootstrap=True` la fórmula se equivoca hasta en 0.044 (unos \$4 400) y en promedio en 0.016. La razón está en con qué filas calculó cada árbol el promedio de sus hojas: cada árbol se ajusta con **su muestra bootstrap**, $n$ filas sacadas con reemplazo. En promedio el 36.8 % de las filas ($\approx e^{-1}$) no aparecen en la muestra de un árbol y otras aparecen dos o más veces. El valor de la hoja es el promedio de las $y$ de **esa muestra**, no de todas las filas de entrenamiento que caen en la hoja. La fórmula de 1.4 cuenta a cada fila una vez, incluidas las que el árbol nunca vio.

Si en el árbol $t$ la fila $i$ aparece $c_{ti}$ veces en la muestra bootstrap, el peso correcto es

$$w_i(x) = \frac{1}{T}\sum_t \frac{c_{ti}\, \mathbf 1\{x_i \in L_t(x)\}}{\sum_j c_{tj}\, \mathbf 1\{x_j \in L_t(x)\}},$$

y con él el error vuelve a ser del orden de $10^{-15}$. scikit-learn implementa el bootstrap justamente así, como pesos de muestra $c_{ti}$; para recuperarlos se usa una función interna (`_generate_sample_indices`), solo para comprobar la explicación.

## 1.5 · Verificación de los cinco modelos

`verificar(w, y_train, pred)` (en `src/bosque.py`) hace `assert` de $\left|\sum_i w_i y_i - \text{pred}\right| < 10^{-8}$. Se aplica a las primeras 20 filas de prueba con los hiperparámetros de la Parte 1.

In [ ]:
modelos = {
    "OLS": LinearRegression(),
    "Ridge (λ = 100)": Ridge(alpha=100),
    "k-NN (k = 30)": KNeighborsRegressor(n_neighbors=30),
    "Árbol (prof. 6)": DecisionTreeRegressor(max_depth=6, random_state=0),
}
for m in modelos.values():
    m.fit(X_train, y_train)
modelos["Bosque"] = bosque

W = {
    "OLS": pesos_ols(X_train, q),
    "Ridge (λ = 100)": pesos_ridge(X_train, q, 100),
    "k-NN (k = 30)": pesos_knn(modelos["k-NN (k = 30)"], q, n),
    "Árbol (prof. 6)": pesos_arbol(modelos["Árbol (prof. 6)"], X_train, q),
    "Bosque": W_bosque,
}
pd.Series({nombre: verificar(W[nombre], y_train, modelos[nombre].predict(q)) for nombre in W},
          name="error máx. en las 20 filas").to_frame()

Tabla para la fila de prueba 7 (posición 7 de `X_test`, contando desde 0):

In [ ]:
fila = 7
pd.DataFrame({nombre: {
    "predicción": modelos[nombre].predict(q.iloc[[fila]])[0],
    "Σ w_i": W[nombre][fila].sum(),
    "pesos ≠ 0": int((W[nombre][fila] != 0).sum()),
    "pesos < 0": int((W[nombre][fila] < 0).sum()),
} for nombre in W}).T

Los cinco modelos pasan el `assert` en las 20 filas, con errores del orden de $10^{-15}$ (redondeo de punto flotante). En la fila 7 los cinco pesos suman 1, pero cambia quién vota y cómo:

- **OLS y ridge** usan las 16 512 filas de entrenamiento y unas 4 100 tienen peso negativo. Por eso predicen casi lo mismo (3.06): con $n$ tan grande, $\lambda = 100$ apenas encoge los coeficientes.
- **k-NN** usa exactamente 30 filas, todas con peso $1/30$.
- **El árbol** usa las 328 filas de la hoja de $x$, todas con el mismo peso.
- **El bosque** usa 388 filas distintas: la unión de las hojas de $x$ en los 200 árboles. Cada hoja tiene solo entre 5 y unas pocas decenas de filas, pero las hojas cambian de árbol a árbol. Los pesos son no negativos y desiguales: pesan más las filas que comparten hoja con $x$ en muchos árboles.

Los modelos locales (árbol y bosque) predicen 3.70–3.72, bastante por encima de OLS: en esta zona la relación no es lineal y el promedio global no la capta.